# The first backtest, on the reference engine

The reference engine of step 2.2 is FEFAT's slow, readable implementation of the fill specification (`docs/spec/fill-semantics.md`), which the Rust engine will have to match fill for fill. This notebook runs a deliberately trivial strategy through it: buy one lot when a 5-minute close crosses above the mean of the last 20 closes, between 08:30 and 15:00 Chicago time, with a stop 12 ticks below the fill and a target 24 ticks above it, and close any position from 15:05 Chicago.

Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and changes nothing in it. Your store holds 1-minute bars and no ticks, so the strategy decides on 5-minute closes and its orders fill on 1-minute bars, each replayed as four pseudo-ticks: the run is at bar fidelity, and every fill is tagged `1m`. The last cell runs the same strategy on ticks, on the repository's synthetic ES sessions.

The reference engine is written to be read, not to be fast: it checks every order at every record or bar, one by one. This month takes under a second on this PC, since the strategy has few orders working at a time; a strategy with many working orders, or a year of ticks, takes minutes, and speed is the Rust engine's job (step 2.6). Profit and loss are before fees, which join runs with the dated fee tables of step 2.9. Above each cell is what it should print on this PC.

## Setup

The strategy and the helpers the cells below share. Expected: the path of your data folder, `%LOCALAPPDATA%\FEFAT\data` unless you set `FEFAT_DATA_DIR`.

In [ ]:
import os
import time
from datetime import date, timedelta
from datetime import time as clock
from pathlib import Path
from typing import Any

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.config import resolve_paths
from fefat.market.bars import build_bars
from fefat.market.calendar import SessionTable, counted
from fefat.market.instruments import Instrument
from fefat.market.layout import Dataset, PartitionRequest
from fefat.market.saved_instruments import instrument_table
from fefat.market.schema import Timeframe, parse_schema_name
from fefat.reference.engine import ReferenceEngine
from fefat.reference.hours import CalendarHours
from fefat.reference.ledger import Trade, trades_of
from fefat.reference.exit_plan import ExitPlan
from fefat.reference.orders import Side
from fefat.reference.settings import RunSettings
from fefat.reference.strategy import DecisionContext, decisions_of
from fefat.reference.tape import Record, tape_of_bars, tape_of_trades
from fefat.reference.timeline import RunResult
from fefat.reference.units import instrument_parameters, money_text

paths = resolve_paths(os.environ)
instruments = instrument_table(paths.data_dir)
MEAN_OF, STOP_TICKS, TARGET_TICKS, SESSIONS = 20, 12, 24, 23
ENTRIES_FROM, ENTRIES_UNTIL, FLAT_BY = clock(8, 30), clock(15, 0), clock(15, 5)
ONE_MINUTE = Timeframe.parse("1m")
TRADE_COLUMNS = {name: pl.Int64 for name in Trade.model_fields} | {
    "side": pl.String,
    "exit_reason": pl.String,
}


def with_signals(bars: pl.DataFrame, zone: str) -> pl.DataFrame:
    """The bars in time order with what the strategy reads at each close: whether the close
    crossed above the mean of the last 20 closes, and the close's time of day in `zone`."""
    mean = pl.col("close").rolling_mean(MEAN_OF)
    closes = pl.from_epoch("ts_close", time_unit="ns").dt.replace_time_zone("UTC")
    return bars.sort("ts_close").with_columns(
        cross_up=(pl.col("close") > mean) & (pl.col("close").shift(1) <= mean.shift(1)),
        local_close=closes.dt.convert_time_zone(zone).dt.time(),
    )


def on_bar(bar: dict[str, Any], context: DecisionContext) -> None:
    """At each 5-minute close: from 15:05, close the position; otherwise, flat and with nothing
    working, buy one lot with its bracket on a cross above the mean inside the entry hours."""
    if context.position != 0 and bar["local_close"] >= FLAT_BY:
        context.flatten()
        return
    entering = ENTRIES_FROM <= bar["local_close"] < ENTRIES_UNTIL and bar["cross_up"]
    if entering and context.position == 0 and not context.has_working_orders():
        bracket = ExitPlan.in_ticks(stop_ticks=STOP_TICKS, target_ticks=TARGET_TICKS)
        context.market(Side.BUY, 1, bracket)


def backtest(
    tape: tuple[Record, ...],
    decisions: pl.DataFrame,
    fidelity: ff.market.Fidelity,
    instrument: Instrument,
    sessions: SessionTable,
) -> tuple[RunResult, float]:
    """The strategy through the reference engine on `tape`, and the seconds it took."""
    hours = CalendarHours.around(sessions, tape[0].starts_at(), tape[-1].available_at())
    parameters = instrument_parameters(instrument)
    engine = ReferenceEngine(RunSettings.for_fidelity(fidelity), parameters, fidelity, hours)
    started = time.perf_counter()
    result = engine.run(tape, decisions_of(decisions), on_bar)
    return result, time.perf_counter() - started


def trade_list(trades: tuple[Trade, ...], zone: str) -> pl.DataFrame:
    """The trades as a chart shows them: times in `zone`, prices in points, and profit and loss
    in the instrument's currency, before fees."""
    frame = pl.DataFrame([trade.model_dump() for trade in trades], schema=TRADE_COLUMNS)
    utc = {
        name: pl.from_epoch(name, time_unit="ns").dt.replace_time_zone("UTC")
        for name in ("entry_ts", "exit_ts")
    }
    return frame.select(
        entry=utc["entry_ts"].dt.convert_time_zone(zone),
        exit=utc["exit_ts"].dt.convert_time_zone(zone),
        side="side",
        entry_price=pl.col("entry_price") / 1e9,
        exit_price=pl.col("exit_price") / 1e9,
        reason="exit_reason",
        pnl=pl.col("pnl") / 1e6,
    )


def show(trades: tuple[Trade, ...], zone: str, currency: str) -> None:
    """Displays every trade, and prints how many there are and their total before fees."""
    with pl.Config(tbl_rows=-1):
        display(trade_list(trades, zone))
    total = money_text(sum(trade.pnl for trade in trades))
    print(f"Trades: {len(trades)}; total before fees: {total} {currency}")


print(f"Data folder: {paths.data_dir}")

## The product and the month

The product is MES when your store holds its 1-minute bars, else the 1-minute bars with the most contracts, as in `01-market-data.ipynb`. The month is the last 23 sessions of the contract that was the front for the most sessions in the last 120 days, so that the run never crosses a roll: positions across a roll arrive in step 2.9.

Expected on this PC: `MES from databento: MESU2026, 23 sessions from 2026-08-12 to 2026-09-14`.

In [ ]:
listed = ff.market.catalog().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
daily = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=120), LAST, fidelity="1m", provider=PROVIDER
).collect()
fronts = (
    daily.group_by("contract")
    .agg(sessions=pl.len(), last=pl.col("trade_date").max())
    .sort("sessions", "last", descending=True)
)
CONTRACT = fronts.item(0, "contract")
held = daily.filter(pl.col("contract") == CONTRACT).get_column("trade_date").sort().tail(SESSIONS)
START, END = held.min(), held.max()
print(f"{ROOT} from {PROVIDER}: {CONTRACT}, {counted(held.len(), 'session')} from {START} to {END}")

## The decisions and the tape

The decisions are the contract's 5-minute closes, with the 20-bar mean and the time of day in Chicago; the tape is its stored 1-minute bars, at the contract's own prices: a minute without trades has no bar, so nothing fills in it.

Expected on this PC: 6,576 five-minute closes, 124 of them crosses inside the entry hours, and 32,880 stored 1-minute bars.

In [ ]:
instrument = instruments.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
five_minutes = ff.market.load(CONTRACT, "5m", START, END, fidelity="1m", provider=PROVIDER)
decisions = with_signals(five_minutes.collect(), zone)
minutes = ff.market.load(CONTRACT, "1m", START, END, fidelity="1m", provider=PROVIDER).collect()
tape = tape_of_bars(minutes)
in_hours = pl.col("local_close").is_between(ENTRIES_FROM, ENTRIES_UNTIL, closed="left")
crosses = decisions.filter(pl.col("cross_up") & in_hours).height
print(f"Decisions: {decisions.height:,} five-minute closes, {crosses:,} crosses in the entry hours")
print(f"Tape: {len(tape):,} stored 1-minute bars of {CONTRACT}")

## The run

The reference engine replays every 1-minute bar as its open, its two extremes, and its close, and when a bar fills an order it replays the bar again with its extremes the other way round: if that would fill otherwise, as when one bar reaches both the stop and the target, the fills are flagged `ambiguous` (F-BAR-05), and the stop is the one taken (F-BAR-02).

Expected on this PC: 224 fills, none of them ambiguous this month, all of fidelity `1m`, in under a second, and no position at the end.

In [ ]:
sessions = instruments.sessions_of(ROOT, paths.cache_dir)
result, seconds = backtest(tape, decisions, ONE_MINUTE, instrument, sessions)
ambiguous = sum(1 for fill in result.fills if "ambiguous" in fill.flags)
fidelities = ", ".join(sorted({str(fill.fidelity) for fill in result.fills})) or "none"
print(f"Fills: {len(result.fills)}, {ambiguous} ambiguous, at fidelity {fidelities}")
print(f"Run: {seconds:.1f} s for {len(tape):,} bars; position at the end: {result.position} lots")

## The trade list

One row per trade: its entry and exit in Chicago time, its prices in points, why it closed, and its profit and loss in US dollars before fees. A trade closes at its `stop` (12 ticks under the entry's fill, filled a tick under that), its `target` (24 ticks over), or at the strategy's `exit` from 15:05. The engine's event log (`result.log.lines()`) has every decision behind each fill, with the rule that priced it.

Expected on this PC: the list of 112 trades, and their total before fees.

In [ ]:
TRADES = trades_of(result.fills)
show(TRADES, zone, instrument.currency)

## Tick fills, on the synthetic ES sessions

Your store holds no ticks, so this cell runs the same strategy on the repository's synthetic ES trades of 2026-03-05 and 03-06, those of ESH2026, which the tests run on: its 5-minute bars are built from the trades, and its orders fill on the trades themselves (F-MKT-02, F-STP-03), so every fill is tagged `trades`. The prices are generated: the cell shows the engine filling on ticks, not a result.

Expected: 143,455 trades of ESH2026, 9 trades of the strategy, and fills of fidelity `trades`.

In [ ]:
REPOSITORY = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "fixtures" / "synthetic" / "MANIFEST.json").is_file()
    ),
    None,
)
if REPOSITORY is None:
    raise ValueError(
        f"Expected the repository's fixtures folder at or above {Path.cwd()}, got none: open "
        "the notebook from the repository."
    )
es = instruments.get("ES")
es_zone = es.session_calendar().hours.time_zone
es_sessions = instruments.sessions_of("ES", paths.cache_dir)
es_trades = Dataset(provider="synthetic", schema_name=parse_schema_name("trades"), root="ES")
request = PartitionRequest(dataset=es_trades, start=date(2026, 3, 5), end=date(2026, 3, 6))
prints = (
    ff.market.Store(REPOSITORY / "fixtures")
    .scan_contracts(request)
    .filter(pl.col("contract") == "ESH2026")
    .drop("contract")
    .sort("ts_recv", "sequence")
    .collect()
)
received_until = prints.get_column("ts_recv").max() + 1
es_bars = build_bars(prints, Timeframe.parse("5m"), es_sessions, received_until)
ticks = tape_of_trades(prints)
tick_fidelity = ff.market.TickFidelity.TRADES
tick_result, tick_seconds = backtest(
    ticks, with_signals(es_bars, es_zone), tick_fidelity, es, es_sessions
)
TICK_TRADES = trades_of(tick_result.fills)
tags = ", ".join(sorted({str(fill.fidelity) for fill in tick_result.fills})) or "none"
print(f"Ticks: {len(ticks):,} trades of ESH2026 in {tick_seconds:.1f} s; fills at fidelity {tags}")
show(TICK_TRADES, es_zone, es.currency)

## Your checks

Pick one trade of the 1-minute run and open the same day on your charting platform, on the contract above, with 5-minute and 1-minute bars in Chicago time:

- the 5-minute bar before the entry closed above the mean of the last 20 closes, after closing at or under it;
- the entry filled at the open of the next 1-minute bar, plus a tick;
- a `stop` exit filled a tick under the stop, 12 ticks under the entry's fill, and a `target` exit at the target, 24 ticks over it.

Step 2.10 runs the same strategy through the Rust engine on the same bars and compares its trade list with this one.